# 확장 백본 비교 실험: MTGNN + Mamba + Chronos-LoRA

**기존 4개 모델** (StockMixer / PatchTST / iTransformer / Chronos zero-shot) + **신규 3개 모델** 비교

| 신규 모델 | 방식 | 기존 대비 차별점 |
|-----------|------|------------------|
| **MTGNN** | Adaptive Graph + Gated TCN | 피처 간 명시적 그래프 구조 자동 학습 |
| **Mamba** | State Space Model (SSM) | 선형 시간복잡도 / 장기 의존성 포착 |
| **Chronos-LoRA** | T5 인코더 LoRA 파인튜닝 | zero-shot → KOSPI 파인튜닝 성능 개선 |

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# peft가 torchao>=0.16.0 요구 (Colab 기본 0.10.0 → 먼저 업그레이드)
!pip install -q "torchao>=0.16.0"
!pip install -q chronos-forecasting einops h5py pyarrow tqdm peft

# mamba-ssm 설치 시도 (실패 시 BiGRU 대체)
import subprocess
_res = subprocess.run(
    ['pip', 'install', '-q', 'mamba-ssm', 'causal-conv1d'],
    capture_output=True, text=True, timeout=300
)
MAMBA_OK = _res.returncode == 0
print(f'Mamba-SSM: {"✓ 설치 완료" if MAMBA_OK else "✗ 설치 실패 → BiGRU 대체"}')

In [ ]:
import gc, warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from pathlib import Path

if MAMBA_OK:
    try:
        from mamba_ssm import Mamba
    except Exception:
        MAMBA_OK = False
        print('Mamba import 실패 → BiGRU 대체')

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/grad_project')
DATA_DIR   = DRIVE_ROOT / 'data'
SEQ_DIR    = DATA_DIR / 'sequences'
H5_PATH    = SEQ_DIR / 'sequences.h5'
MODEL_DIR  = DRIVE_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN    = 60
N_PATCHES  = 12
PATCH_LEN  = 5
N_FEATURES = 20

BATCH_SIZE = 512
N_EPOCHS   = 10
LR         = 1e-3
MAX_TRAIN  = 100_000
PATIENCE   = 3

assert H5_PATH.exists(), f'파일 없음: {H5_PATH}'
print(f'sequences.h5: {H5_PATH.stat().st_size/1e9:.2f} GB')

In [ ]:
class KospiH5Dataset(Dataset):
    def __init__(self, h5_path, split, max_samples=None):
        with h5py.File(h5_path, 'r') as f:
            n = f[split]['y_ret'].shape[0]
            if max_samples and n > max_samples:
                step = n // max_samples
                sl   = slice(0, n, step)
            else:
                sl = slice(None)
            self.X      = f[split]['X'][sl][:max_samples or n]
            self.X_flat = f[split]['X_flat'][sl][:max_samples or n]
            self.y_ret  = f[split]['y_ret'][sl][:max_samples or n].astype(np.float32)
            self.y_dir  = f[split]['y_dir'][sl][:max_samples or n].astype(np.int64)
        print(f'{split:5s}: {len(self.y_ret):>8,}개 | 상승: {self.y_dir.mean():.3f}')

    def __len__(self): return len(self.y_ret)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X[i]),
            torch.from_numpy(self.X_flat[i]),
            torch.tensor(self.y_ret[i]),
            torch.tensor(self.y_dir[i]),
        )

print('데이터 로드 중...')
train_ds = KospiH5Dataset(H5_PATH, 'train', max_samples=MAX_TRAIN)
val_ds   = KospiH5Dataset(H5_PATH, 'val')
test_ds  = KospiH5Dataset(H5_PATH, 'test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('완료')

In [ ]:
def compute_metrics(y_true, y_pred):
    dir_acc  = ((y_true > 0) == (y_pred > 0)).mean()
    spear, _ = spearmanr(y_true, y_pred)
    mae      = np.abs(y_true - y_pred).mean()
    rmse     = np.sqrt(((y_true - y_pred) ** 2).mean())
    return {
        'dir_acc':  round(float(dir_acc), 4),
        'spearman': round(float(spear), 4),
        'mae':      round(float(mae), 6),
        'rmse':     round(float(rmse), 6),
    }


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for x_patch, x_flat, y_ret, _ in loader:
        x_patch, x_flat, y_ret = x_patch.to(device), x_flat.to(device), y_ret.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_patch, x_flat), y_ret)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    for x_patch, x_flat, y_ret, _ in loader:
        pred = model(x_patch.to(device), x_flat.to(device)).cpu().numpy()
        preds.append(pred)
        trues.append(y_ret.numpy())
    return compute_metrics(np.concatenate(trues), np.concatenate(preds))


def train_model(model, name, train_loader, val_loader, test_loader,
                lr=LR, n_epochs=N_EPOCHS):
    model = model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    total_all = sum(p.numel() for p in model.parameters())
    total_tr  = sum(p.numel() for p in trainable)
    print(f'[{name}] 전체: {total_all:,} | 학습: {total_tr:,} ({100*total_tr/total_all:.1f}%)')

    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.MSELoss()

    best_mae, no_improve, best_state = float('inf'), 0, None
    history = []

    for epoch in range(1, n_epochs + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_m   = evaluate(model, val_loader, DEVICE)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': tr_loss, **val_m})
        print(f'[{name}] ep{epoch:02d} | loss={tr_loss:.5f} | '
              f'val_dir={val_m["dir_acc"]:.4f} | val_mae={val_m["mae"]:.5f}')

        if val_m['mae'] < best_mae:
            best_mae   = val_m['mae']
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    test_m = evaluate(model.to(DEVICE), test_loader, DEVICE)
    print(f'\n[{name}] Test: {test_m}\n')
    return model, test_m, history

In [ ]:
# 이전 backbone_comparison_colab.ipynb 결과 로드
prev_path = DATA_DIR / 'backbone_results.csv'
if prev_path.exists():
    prev_df = pd.read_csv(prev_path, index_col=0)
    print('이전 결과 로드 완료:')
else:
    # backbone_comparison_colab.ipynb 미실행 시 하드코딩 폴백
    prev_df = pd.DataFrame({
        'StockMixer':          {'dir_acc': 0.5033, 'spearman': 0.0072, 'mae': 0.04515, 'rmse': 0.11202},
        'PatchTST':            {'dir_acc': 0.4960, 'spearman': 0.0096, 'mae': 0.04392, 'rmse': 0.11191},
        'iTransformer':        {'dir_acc': 0.5057, 'spearman': 0.0056, 'mae': 0.04330, 'rmse': 0.11101},
        'Chronos (zero-shot)': {'dir_acc': 0.5096, 'spearman': 0.0192, 'mae': None,    'rmse': None},
    }).T
    print('이전 결과 (하드코딩 폴백):')

print(prev_df.to_string())
print(f'\n랜덤 베이스라인: dir_acc=0.5000, spearman=0.0000')

## 1. MTGNN (Multivariate Temporal Graph Neural Network)

**논문**: Wu et al., 2020 — "Connecting the Dots: Multivariate Time Series Forecasting with Graph Neural Networks"

### 핵심 아이디어
20개 기술적 지표 = **그래프 노드**, 지표 간 상관관계 = **엣지 (자동 학습)**

```
[Adaptive Graph Learner]  E1 @ E2ᵀ → softmax → A (20×20 인접 행렬)
         ↓
[Gated TCN × 3]           각 노드의 60일 시계열에 dilated conv (d=1,2,4)
         ↓
[Graph Conv × 3]          A를 이용해 이웃 노드 정보 집계 (MACD→RSI 등)
         ↓
[Mean Pool + Head]        20×d_model → 128 → 1
```

**iTransformer와 차이**: iTransformer는 Attention으로 피처 관계 학습; MTGNN은 **명시적 그래프 구조**로 학습 → 해석 가능한 인접 행렬 시각화 가능

In [ ]:
class AdaptiveGraphLearner(nn.Module):
    """학습 가능한 노드 임베딩 → 피처 그래프 인접 행렬 자동 생성"""
    def __init__(self, n_nodes, emb_dim=10):
        super().__init__()
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)

    def forward(self):
        # (N, N) — softmax(ReLU(E1 @ E2ᵀ))
        return torch.softmax(torch.relu(self.E1 @ self.E2.T), dim=-1)


class MTGNN(nn.Module):
    """
    Adaptive Graph + Gated Dilated TCN 백본

    N=20 기술적 지표를 그래프 노드로 취급:
    - TCN: 각 노드의 시계열 패턴 학습 (temporal)
    - Graph Conv: 인접 행렬로 노드 간 정보 전파 (cross-feature)
    """
    def __init__(self, seq_len=60, n_features=20, d_model=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.n_features = n_features

        # (B, 1, N, T) → (B, d_model, N, T)
        self.input_proj = nn.Conv2d(1, d_model, kernel_size=(1, 1))

        # Gated TCN: dilation 1, 2, 4
        dilations = [1, 2, 4][:n_layers]
        self.filter_convs = nn.ModuleList([
            nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d))
            for d in dilations
        ])
        self.gate_convs = nn.ModuleList([
            nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d))
            for d in dilations
        ])
        self.res_convs  = nn.ModuleList([nn.Conv2d(d_model, d_model, 1) for _ in dilations])
        self.gc_weights = nn.ModuleList([nn.Linear(d_model, d_model, bias=False) for _ in dilations])
        self.norms      = nn.ModuleList([nn.LayerNorm(d_model) for _ in dilations])

        self.graph = AdaptiveGraphLearner(n_features)
        self.drop  = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(n_features * d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x_patch, x_flat):
        B, T, N = x_flat.shape

        # (B, T, N) → (B, 1, N, T)
        x = x_flat.permute(0, 2, 1).unsqueeze(1)
        x = self.input_proj(x)  # (B, d, N, T)

        A = self.graph()  # (N, N) — 학습 가능 인접 행렬

        for fc, gc_c, rc, gc_w, norm in zip(
            self.filter_convs, self.gate_convs, self.res_convs, self.gc_weights, self.norms
        ):
            residual = x
            T_cur = x.size(-1)

            # Gated TCN: (B, d, N, T)
            h = (torch.tanh(fc(x)[..., :T_cur]) *
                 torch.sigmoid(gc_c(x)[..., :T_cur]))

            # Graph conv: 각 시간 스텝에서 노드 간 정보 집계
            # h: (B, d, N, T) → (B, T, N, d)
            h_p  = h.permute(0, 3, 2, 1)
            # einsum: output node n ← sum over m of A[n,m] * h[m]
            h_gc = torch.einsum('nm,btmc->btnc', A, h_p)
            h_gc = gc_w(h_gc).permute(0, 3, 2, 1)  # (B, d, N, T)

            # Residual + LayerNorm (over d dim)
            combined = self.drop(h_gc) + rc(residual)  # (B, d, N, T)
            combined = combined.permute(0, 3, 2, 1)    # (B, T, N, d)
            x = norm(combined).permute(0, 3, 2, 1)     # (B, d, N, T)

        # Global avg pool over T → flatten N*d
        x = x.mean(-1)                      # (B, d, N)
        x = x.permute(0, 2, 1).flatten(1)  # (B, N*d)
        return self.head(x).squeeze(-1)

    @torch.no_grad()
    def get_adjacency(self):
        """학습된 피처 간 인접 행렬 반환 (시각화용)"""
        return self.graph().cpu().numpy()


_m = MTGNN(SEQ_LEN, N_FEATURES)
print(f'MTGNN 파라미터: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()

mtgnn_model, mtgnn_metrics, mtgnn_hist = train_model(
    MTGNN(SEQ_LEN, N_FEATURES, d_model=64, n_layers=3),
    'MTGNN', train_loader, val_loader, test_loader
)
torch.save(mtgnn_model.state_dict(), MODEL_DIR / 'mtgnn_best.pt')
print('MTGNN 저장 완료')

## 2. Mamba (State Space Model)

**논문**: Gu & Dao, 2023 — "Mamba: Linear-Time Sequence Modeling with Selective State Spaces"

### 핵심 아이디어
Transformer의 O(T²) Attention → **O(T) 선형 선택적 상태공간 모델**

```
입력: (B, T=60, N=20)
  ↓ Linear embed → (B, 60, d_model=128)
  ↓ Mamba Block × 4    ← 각 블록: LN + SSM + residual
      SSM: 선택적 파라미터(Δ, B, C)로 현재 입력에 따라 상태 업데이트
  ↓ Mean pool → (B, d_model)
  ↓ Head → (B, 1)
```

**Transformer 대비 장점**: 긴 시계열에서 기울기 소실 없이 장기 의존성 포착  
**설치 실패 시**: BiGRU로 대체 (같은 인터페이스, SSM 근사)

In [ ]:
# Mamba 사용 가능 여부 표시
print(f'Mamba-SSM: {"사용" if MAMBA_OK else "사용 불가 → BiGRU 대체"}')
MODEL_LABEL_MAMBA = 'Mamba' if MAMBA_OK else 'BiGRU (Mamba 대체)'


class MambaBlock(nn.Module):
    """Mamba SSM 블록 (mamba-ssm 없으면 BiGRU 대체)"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

        if MAMBA_OK:
            self.ssm  = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
            self._use_mamba = True
        else:
            # Bidirectional GRU: d_model → d_model/2 × 2 = d_model
            self.rnn  = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
            self.gate = nn.Sequential(nn.Linear(d_model, d_model), nn.Sigmoid())
            self._use_mamba = False

    def forward(self, x):
        # x: (B, T, d)
        h = self.norm(x)
        if self._use_mamba:
            h = self.ssm(h)
        else:
            h_rnn, _ = self.rnn(h)
            h = h_rnn * self.gate(h)
        return x + self.drop(h)


class MambaModel(nn.Module):
    """
    Mamba (SSM) 기반 시계열 백본

    Transformer의 피처 간 Attention 대신:
    - Mamba: 시간축 selective state space (장기 의존성 O(T) 처리)
    - 20개 피처를 하나의 d_model 벡터로 임베딩 후 시간 축 처리
    """
    def __init__(self, seq_len=60, n_features=20, d_model=128, n_layers=4, dropout=0.1):
        super().__init__()
        self.embed  = nn.Linear(n_features, d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(d_model, dropout) for _ in range(n_layers)
        ])
        self.norm   = nn.LayerNorm(d_model)
        self.head   = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_patch, x_flat):
        # x_flat: (B, T=60, N=20)
        x = self.embed(x_flat)       # (B, T, d)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x).mean(dim=1)  # global avg pool over T: (B, d)
        return self.head(x).squeeze(-1)


_m = MambaModel(SEQ_LEN, N_FEATURES)
print(f'{MODEL_LABEL_MAMBA} 파라미터: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()

mamba_loader_train = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
mamba_loader_val   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
mamba_loader_test  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

mamba_model, mamba_metrics, mamba_hist = train_model(
    MambaModel(SEQ_LEN, N_FEATURES, d_model=128, n_layers=4),
    MODEL_LABEL_MAMBA, mamba_loader_train, mamba_loader_val, mamba_loader_test
)
torch.save(mamba_model.state_dict(), MODEL_DIR / 'mamba_best.pt')
print(f'{MODEL_LABEL_MAMBA} 저장 완료')

## 3. Chronos-LoRA (Fine-tuned Chronos T5 Encoder)

**기존**: Chronos zero-shot (학습 데이터 전혀 없이 추론)  
**신규**: LoRA로 KOSPI 데이터에 파인튜닝 → 도메인 적응

### LoRA (Low-Rank Adaptation) 란?

```
기존 Attention 가중치 W (512×512) — 동결
    + 저랭크 행렬 W_A (512×8) @ W_B (8×512) — 학습
총 파라미터 ~60M 중 ~1M만 학습 (1.7%)
```

### 아키텍처

```
입력: x_flat[:, :, 0] = Adj_Close (B, T=60)
  ↓ Chronos 토크나이저 (MeanScaleUniformBins): float → 정수 빈 인덱스
  ↓ T5 인코더 + LoRA (Q, V 프로젝션에 적용): (B, L, 512)
  ↓ Masked Mean Pooling: (B, 512)
  ↓ LayerNorm → Linear(512, 128) → GELU → Linear(128, 1)
```

**핵심 질문**: KOSPI 파인튜닝이 zero-shot(50.96%) 대비 방향성을 개선하는가?

In [ ]:
try:
    from peft import LoraConfig, get_peft_model
    PEFT_OK = True
except ImportError:
    PEFT_OK = False
    print('peft 없음: !pip install peft 실행 필요')


class ChronosLoRAModel(nn.Module):
    """
    Chronos T5 인코더 + LoRA 파인튜닝 + 회귀 헤드

    - Chronos 토크나이저: Adj_Close 시계열 → 정수 토큰 (MeanScaleUniformBins)
    - T5 인코더: LoRA 적용 (Q, V에만, r=8)
    - 나머지 파라미터 동결 → 빠른 파인튜닝
    """
    def __init__(self, model_name='amazon/chronos-t5-small', lora_r=8):
        super().__init__()
        assert PEFT_OK, 'peft 설치 필요'
        from chronos import ChronosPipeline

        pipeline = ChronosPipeline.from_pretrained(
            model_name, device_map='cpu', torch_dtype=torch.float32
        )
        self.tokenizer = pipeline.tokenizer
        t5 = pipeline.model.model  # T5ForConditionalGeneration

        lora_cfg = LoraConfig(
            r=lora_r,
            lora_alpha=lora_r * 2,
            target_modules=['q', 'v'],  # T5Attention 내 Q, V 프로젝션
            lora_dropout=0.05,
            bias='none',
        )
        self.t5   = get_peft_model(t5, lora_cfg)
        d_enc     = t5.config.d_model  # chronos-t5-small: 512

        # 회귀 헤드 (LoRA + head만 학습)
        self.head = nn.Sequential(
            nn.LayerNorm(d_enc),
            nn.Linear(d_enc, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
        # 헤드 초기화
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.t5.print_trainable_parameters()

    def _get_encoder(self):
        """peft 버전 차이 대응용 encoder 접근자"""
        try:
            return self.t5.encoder
        except AttributeError:
            return self.t5.base_model.model.encoder

    def forward(self, x_patch, x_flat):
        device = x_flat.device

        # Chronos 토크나이저는 CPU에서 실행
        adj_close = x_flat[:, :, 0].cpu()  # (B, 60) float
        input_ids, attn_mask, _ = self.tokenizer.context_input_transform(adj_close)
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)

        enc_out = self._get_encoder()(
            input_ids=input_ids,
            attention_mask=attn_mask,
        ).last_hidden_state  # (B, L, d_enc)

        # Masked mean pooling
        mask   = attn_mask.unsqueeze(-1).float()
        pooled = (enc_out * mask).sum(1) / mask.sum(1).clamp(min=1)  # (B, d_enc)

        return self.head(pooled).squeeze(-1)


if PEFT_OK:
    _m = ChronosLoRAModel()
    print(f'Chronos-LoRA 전체 파라미터: {sum(p.numel() for p in _m.parameters()):,}')
    del _m
else:
    print('peft 없어서 Chronos-LoRA 건너뜀')

In [ ]:
%%time
if not PEFT_OK:
    print('peft 없음 → Chronos-LoRA 건너뜀')
    chronos_lora_metrics = None
else:
    gc.collect()
    torch.cuda.empty_cache()

    # 배치 크기 작게 (T5 인코더 + 토크나이저 오버헤드)
    CL_BATCH = 64
    cl_train = DataLoader(train_ds, batch_size=CL_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
    cl_val   = DataLoader(val_ds,   batch_size=CL_BATCH, shuffle=False, num_workers=2, pin_memory=True)
    cl_test  = DataLoader(test_ds,  batch_size=CL_BATCH, shuffle=False, num_workers=2, pin_memory=True)

    chronos_lora_model, chronos_lora_metrics, chronos_lora_hist = train_model(
        ChronosLoRAModel(lora_r=8),
        'Chronos-LoRA', cl_train, cl_val, cl_test,
        lr=2e-4,   # 사전학습 모델 파인튜닝 → 낮은 LR
        n_epochs=15,
    )
    # LoRA 어댑터만 저장
    chronos_lora_model.t5.save_pretrained(str(MODEL_DIR / 'chronos_lora_adapters'))
    torch.save(chronos_lora_model.head.state_dict(), MODEL_DIR / 'chronos_lora_head.pt')
    print('Chronos-LoRA 저장 완료')

## 4. 확장 비교 결과 (7개 모델)

| 모델 | 방식 | 특징 |
|------|------|------|
| StockMixer | MLP-Mixer | 경량 베이스라인 |
| PatchTST | 패치 Transformer | 시간축 Attention |
| iTransformer | 역전 Transformer | 피처 간 Attention |
| Chronos (zero-shot) | T5 대형 사전학습 | 학습 없이 추론 |
| **MTGNN** | Adaptive Graph + TCN | **명시적 피처 그래프** |
| **Mamba** | State Space Model | **선형 시간복잡도** |
| **Chronos-LoRA** | T5 + LoRA 파인튜닝 | **도메인 적응** |

In [ ]:
# 신규 결과 수집
new_results = {}
new_results['MTGNN'] = mtgnn_metrics
new_results[MODEL_LABEL_MAMBA] = mamba_metrics
if chronos_lora_metrics is not None:
    new_results['Chronos-LoRA'] = chronos_lora_metrics

new_df = pd.DataFrame(new_results).T

# 이전 결과와 합치기
all_df = pd.concat([prev_df, new_df])

# dir_acc 기준으로 정렬 (내림차순)
all_df = all_df.sort_values('dir_acc', ascending=False)

print('=' * 65)
print('확장 백본 비교 실험 결과 (Test Set)')
print('=' * 65)
print(all_df.to_string())
print()
print(f'랜덤 베이스라인: dir_acc=0.5000')
print(f'\n▶ 방향성 1위: {all_df["dir_acc"].idxmax()}')
print(f'▶ Spearman 1위: {all_df["spearman"].idxmax()}')
print(f'\n▶ 신규 모델 중 최고 dir_acc: '
      f'{new_df["dir_acc"].idxmax()} ({new_df["dir_acc"].max():.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models  = all_df.index.tolist()
n       = len(models)
palette = plt.cm.tab10(np.linspace(0, 0.7, n))

# 기존 모델 vs 신규 모델 색상 구분
new_model_names = list(new_results.keys())
colors = [
    '#e15759' if m in new_model_names else '#4e79a7'  # 빨강=신규, 파랑=기존
    for m in models
]

for ax, col, title, ref_val, ref_label in [
    (axes[0], 'dir_acc',  'Directional Accuracy',  0.5,  '랜덤(50%)'),
    (axes[1], 'spearman', 'Spearman Correlation',  0.0,  '무상관(0)'),
    (axes[2], 'mae',      'MAE (낮을수록 좋음)',    None, None),
]:
    vals = pd.to_numeric(all_df[col], errors='coerce')
    valid_mask = vals.notna()
    valid_models = [m for m, v in zip(models, valid_mask) if v]
    valid_vals   = vals[valid_mask].values
    valid_colors = [colors[i] for i, v in enumerate(valid_mask) if v]

    bars = ax.bar(valid_models, valid_vals, color=valid_colors, alpha=0.85,
                  edgecolor='k', linewidth=0.6)
    if ref_val is not None:
        ax.axhline(ref_val, color='gray', lw=1.2, ls='--', label=ref_label)
        ax.legend(fontsize=8)
    for bar, v in zip(bars, valid_vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.0002 if col == 'mae' else 0.001),
                f'{v:.4f}', ha='center', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlim(-0.7, len(valid_models) - 0.3)

# 범례 설명
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#4e79a7', label='기존 모델 (backbone_comparison)'),
    Patch(facecolor='#e15759', label='신규 모델 (이번 실험)'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.02), fontsize=10)

plt.suptitle('확장 백본 비교 실험: 7개 모델', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'extended_backbone_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# MTGNN 학습된 인접 행렬 시각화
FEATURE_NAMES = [
    'Adj_Close', 'Open', 'High', 'Low', 'Volume',
    'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
    'MACD', 'MACD_signal', 'MACD_hist', 'RSI_14',
    'BB_upper', 'BB_lower', 'BB_width',
    'Volume_ratio', 'Return_1d', 'Volatility_20d', 'ATR_14'
]

fig, ax = plt.subplots(figsize=(8, 7))
A_learned = mtgnn_model.get_adjacency()
im = ax.imshow(A_learned, cmap='YlOrRd', vmin=0, aspect='auto')
ax.set_xticks(range(N_FEATURES))
ax.set_yticks(range(N_FEATURES))
ax.set_xticklabels(FEATURE_NAMES, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURE_NAMES, fontsize=8)
ax.set_title('MTGNN 학습된 피처 인접 행렬 (A[n→m])', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(DATA_DIR / 'mtgnn_adjacency_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 피처별 out-degree (다른 피처에 얼마나 영향 주는가)
out_degree = A_learned.sum(axis=1)
rank_df = (
    pd.DataFrame({'feature': FEATURE_NAMES, 'out_degree': out_degree})
    .sort_values('out_degree', ascending=False)
    .reset_index(drop=True)
)
print('\nMTGNN 피처 Out-Degree 순위 (영향력이 큰 피처):')
print(rank_df.to_string())

In [ ]:
# 학습 곡선 비교
all_hist = {
    'MTGNN': mtgnn_hist,
    MODEL_LABEL_MAMBA: mamba_hist,
}
if chronos_lora_metrics is not None:
    all_hist['Chronos-LoRA'] = chronos_lora_hist

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, hist in all_hist.items():
    ep = [h['epoch'] for h in hist]
    axes[0].plot(ep, [h['train_loss'] for h in hist], label=name, marker='o', ms=4)
    axes[1].plot(ep, [h['dir_acc'] for h in hist],    label=name, marker='o', ms=4)

axes[0].set_title('Train Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].axhline(0.5, color='gray', ls='--', lw=1.2, label='랜덤(50%)')
axes[1].set_title('Val Directional Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'extended_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 확장 결과 저장
all_df.to_csv(DATA_DIR / 'backbone_results_extended.csv', encoding='utf-8-sig')
print('저장 완료: backbone_results_extended.csv')
print()
print('생성 파일:')
print('  models/mtgnn_best.pt              — MTGNN 가중치')
print(f'  models/mamba_best.pt              — {MODEL_LABEL_MAMBA} 가중치')
if chronos_lora_metrics is not None:
    print('  models/chronos_lora_adapters/     — LoRA 어댑터')
    print('  models/chronos_lora_head.pt       — 회귀 헤드')
print('  data/backbone_results_extended.csv — 7모델 비교 결과')
print('  data/extended_backbone_comparison.png')
print('  data/mtgnn_adjacency_matrix.png')
print('  data/extended_learning_curves.png')
print()
print('=' * 55)
print('▶ 다음 단계: hybrid_model_colab.ipynb')
print('  Chronos T5 인코더 + iTransformer 하이브리드')
print('  → rl_embeddings.h5 생성 (RL 파트 팀원 전달)')
print('=' * 55)


## 5. Ranking Loss 실험 — ListNet

### 문제 정의
MSE 손실로 학습하면 **절댓값 예측 오차**를 최소화하지만,
실제 목표는 내일 수익률이 **높은 종목 순서대로 잘 맞추는 것** (Spearman ↑).

→ **ListNet (Cao et al., 2007)** — 순위 분포의 Cross-Entropy 최소화

```
L_listnet = -Σ softmax(y_true / τ) × log(softmax(ŷ / τ))
L_total   = L_listnet + λ × MSE(ŷ, y)   (λ=0.3, τ=0.5)
```

### 비교 실험 (2×2)

| | MSE Loss | ListNet Loss |
|---|---|---|
| **MTGNN** | 기준 (이전 결과) | 개선 여부 확인 |
| **iTransformer** | 기준 (이전 결과) | 개선 여부 확인 |

> **가설**: ListNet으로 학습하면 Spearman ρ가 개선된다 (MAE는 소폭 증가 가능)


In [ ]:

# ── 5b. iTransformer 재정의 (Ranking 실험용) + ListNetLoss + train_model_v2 ─────

# iTransformer (피처 간 Attention — 비교군으로 재사용)
class iTransformer4Ranking(nn.Module):
    """iTransformer: 피처 축 self-attention (Ranking 실험 비교용)"""
    def __init__(self, seq_len=60, n_features=20, d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(seq_len, d_model)  # 각 피처의 시계열 → 임베딩
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(n_features * d_model, 128),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x_patch, x_flat):
        # x_flat: (B, T=60, N=20) → (B, N=20, T=60)
        x = x_flat.permute(0, 2, 1)
        x = self.embed(x)           # (B, N, d_model)
        x = self.encoder(x)         # (B, N, d_model) — 피처 간 Attention
        return self.head(x.flatten(1)).squeeze(-1)


# ListNet Ranking Loss
class ListNetLoss(nn.Module):
    """
    ListNet (Cao et al., 2007) — 순위 분포의 KL divergence 최적화

    MSE는 예측값 절댓값 오차를 최소화 → 상승/하락 극단 케이스에 집중
    ListNet은 softmax(pred) ≈ softmax(true_return) → 순위 분포 자체를 학습
      → Spearman 상관계수 직접 개선 효과

    temp: softmax 온도 (낮을수록 상위 종목에 집중)
    mse_weight: MAE 회귀 오차 보조 항 (0이면 순수 ListNet)
    """
    def __init__(self, temp: float = 0.5, mse_weight: float = 0.3):
        super().__init__()
        self.temp       = temp
        self.mse_weight = mse_weight
        self.mse        = nn.MSELoss()

    def forward(self, scores: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # 표준화 후 softmax → 스케일 불변 (수익률이 작은 경우 softmax 포화 방지)
        t_norm = targets / targets.std().clamp(min=1e-6)
        s_norm = scores  / (scores.std().clamp(min=1e-6) + 1e-8)

        P_true = torch.softmax(t_norm / self.temp, dim=0)  # (B,)
        P_pred = torch.softmax(s_norm / self.temp, dim=0)  # (B,)

        # Cross-entropy: H(P_true, P_pred) = -Σ P_true * log(P_pred)
        rank_loss = -(P_true * torch.log(P_pred + 1e-8)).sum()
        return rank_loss + self.mse_weight * self.mse(scores, targets)


# 빠른 smoke-test
_loss = ListNetLoss(temp=0.5, mse_weight=0.3)
_pred = torch.randn(16)
_tgt  = torch.randn(16)
_out  = _loss(_pred, _tgt)
assert _out.item() > 0, "ListNetLoss sanity check 실패"
print(f"ListNetLoss 검증 완료 (sample loss={_out.item():.4f})")
del _loss, _pred, _tgt, _out

_it = iTransformer4Ranking(SEQ_LEN, N_FEATURES, d_model=64)
print(f"iTransformer4Ranking 파라미터: {sum(p.numel() for p in _it.parameters()):,}")
del _it


In [ ]:

# ── 5c. 4가지 실험 수행 ───────────────────────────────────────────────────────
# (MTGNN + MSE) / (MTGNN + ListNet) / (iTransformer + MSE) / (iTransformer + ListNet)
ranking_results = {}

def run_ranking_experiment(model_cls, model_kwargs, criterion, exp_name,
                           lr=LR, n_epochs=N_EPOCHS):
    """ListNet 또는 MSE criterion으로 모델 학습 후 결과 반환"""
    model = model_cls(**model_kwargs).to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_mae, no_improve, best_state = float('inf'), 0, None
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        total = 0.0
        for x_patch, x_flat, y_ret, _ in train_loader:
            x_patch, x_flat, y_ret = (x_patch.to(DEVICE),
                                       x_flat.to(DEVICE),
                                       y_ret.to(DEVICE))
            optimizer.zero_grad()
            pred = model(x_patch, x_flat)
            loss = criterion(pred, y_ret)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()

        val_m = evaluate(model, val_loader, DEVICE)
        scheduler.step()
        history.append({'epoch': epoch, **val_m})
        print(f'[{exp_name}] ep{epoch:02d} | loss={total/len(train_loader):.5f} | '
              f'val_spear={val_m["spearman"]:.4f} | val_dir={val_m["dir_acc"]:.4f}')

        if val_m['mae'] < best_mae:
            best_mae   = val_m['mae']
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    test_m = evaluate(model.to(DEVICE), test_loader, DEVICE)
    print(f'\n[{exp_name}] Test: {test_m}\n')
    return test_m, history

gc.collect()
torch.cuda.empty_cache()
mse_crit     = nn.MSELoss()
listnet_crit = ListNetLoss(temp=0.5, mse_weight=0.3)

MTGNN_KWARGS  = dict(seq_len=SEQ_LEN, n_features=N_FEATURES, d_model=64, n_layers=3)
ITRANS_KWARGS = dict(seq_len=SEQ_LEN, n_features=N_FEATURES, d_model=64, n_heads=4, n_layers=3)

print('=' * 60)
print('실험 1: MTGNN + MSE')
print('=' * 60)
ranking_results['MTGNN + MSE'],    _ = run_ranking_experiment(
    MTGNN, MTGNN_KWARGS, mse_crit, 'MTGNN + MSE')

print('=' * 60)
print('실험 2: MTGNN + ListNet')
print('=' * 60)
ranking_results['MTGNN + ListNet'], _ = run_ranking_experiment(
    MTGNN, MTGNN_KWARGS, listnet_crit, 'MTGNN + ListNet')

print('=' * 60)
print('실험 3: iTransformer + MSE')
print('=' * 60)
ranking_results['iTransformer + MSE'],    _ = run_ranking_experiment(
    iTransformer4Ranking, ITRANS_KWARGS, mse_crit, 'iTransformer + MSE')

print('=' * 60)
print('실험 4: iTransformer + ListNet')
print('=' * 60)
ranking_results['iTransformer + ListNet'], _ = run_ranking_experiment(
    iTransformer4Ranking, ITRANS_KWARGS, listnet_crit, 'iTransformer + ListNet')


In [ ]:

# ── 5d. 결과 비교 및 시각화 ─────────────────────────────────────────────────
rows = []
for name, m in ranking_results.items():
    rows.append({'Model': name, 'dir_acc': m['dir_acc'],
                 'spearman': m['spearman'], 'mae': m['mae']})
df_rank = pd.DataFrame(rows).set_index('Model')
print(df_rank.round(4).to_string())

# ── Spearman 개선량 계산 ──────────────────────────────────────────────────────
delta = (
    df_rank.loc['MTGNN + ListNet', 'spearman'] -
    df_rank.loc['MTGNN + MSE',    'spearman']
)
delta_i = (
    df_rank.loc['iTransformer + ListNet', 'spearman'] -
    df_rank.loc['iTransformer + MSE',     'spearman']
)
print(f"\n▶ MTGNN: ListNet vs MSE 스피어만 변화  {delta:+.4f}")
print(f"▶ iTransformer: ListNet vs MSE 스피어만 변화  {delta_i:+.4f}")

# ── 시각화 ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MSE vs ListNet Ranking Loss', fontsize=13, fontweight='bold')

metrics_def = [
    ('dir_acc',  'Directional Accuracy', True),
    ('spearman', 'Spearman ρ',           True),
    ('mae',      'MAE (낮을수록 좋음)',   False),
]
bar_colors = ['#4e79a7', '#e15759', '#4e79a7', '#e15759']  # MSE=파랑, ListNet=빨강

for ax, (col, title, higher_better) in zip(axes, metrics_def):
    vals = df_rank[col].values
    bars = ax.bar(range(len(df_rank)), vals, color=bar_colors, alpha=0.85,
                  edgecolor='k', linewidth=0.6)
    ax.set_xticks(range(len(df_rank)))
    ax.set_xticklabels(df_rank.index, rotation=25, ha='right', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + abs(bar.get_height()) * 0.015,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#4e79a7', label='MSE Loss'),
              Patch(facecolor='#e15759', label='ListNet Ranking Loss')]
fig.legend(handles=legend_els, loc='upper right', fontsize=10)
plt.tight_layout()

save_path = DATA_DIR / 'ranking_loss_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

# ── 전체 CSV 업데이트 ─────────────────────────────────────────────────────────
df_all_ext = pd.read_csv(DATA_DIR / 'backbone_results_extended.csv', index_col=0)
df_all_ext = pd.concat([df_all_ext, df_rank[~df_rank.index.isin(df_all_ext.index)]])
df_all_ext = df_all_ext.sort_values('spearman', ascending=False)
df_all_ext.to_csv(DATA_DIR / 'backbone_results_extended.csv', encoding='utf-8-sig')
print(f"\n[전체 순위 (spearman 기준)]")
print(df_all_ext[['dir_acc', 'spearman', 'mae']].round(4).to_string())
